# Aggregation, joins and SQL from Python

This notebook covers the second block. The housing analyst from block 1 now asks questions that need many rows at once: how many listings and what average price per district, how many reviews per district once the review table is combined with the listings, which district is the most expensive *for each room type*, and how did the number of reviews develop month by month? Aggregation, joins, common table expressions and window functions answer them. The last section moves the queries into Python programs: connections, cursors and parameters, results as pandas DataFrames, and a script that answers a question from the command line.

```mermaid
flowchart LR
    T["tables<br/>listings, calendar, reviews"] --> J["JOIN<br/>combine by key"]
    J --> G["GROUP BY / HAVING<br/>one row per group"]
    G --> W["window functions<br/>rank, running total, LAG"]
    W --> P["Python: sqlite3,<br/>pandas.read_sql_query"]
    P --> S["script, chart, model"]
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

In [2]:
import sqlite3
import subprocess
import sys

import pandas as pd

DB = Path("case-study/data/airbnb.sqlite")
if not DB.exists():                     # build the course database once from the Parquet files
    subprocess.run([sys.executable, "sessions/03-sql/workbooks/make_airbnb_sqlite.py"], check=True)
con = sqlite3.connect(DB)


def q(sql, params=()):
    """Run an SQL query on the course database and return the result as a pandas DataFrame."""
    return pd.read_sql_query(sql, con, params=params)

## Aggregation with GROUP BY and HAVING

### Concept

An **aggregate function** reduces many values to one: `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`. `COUNT(DISTINCT x)` counts the different values. Without `GROUP BY`, an aggregate summarises the whole table in one row. `GROUP BY` splits the rows into groups with equal values of the grouping columns and computes the aggregates per group; the result has one row per group. Every column in `SELECT` must either be a grouping column or be inside an aggregate. (SQLite does not enforce this rule and returns the value of some row of the group; PostgreSQL refuses the query. Follow the rule anyway.)

`WHERE` filters **rows before** grouping; `HAVING` filters **groups after** aggregation, for example "only districts with at least 300 listings". `FILTER (WHERE ...)` after an aggregate restricts it to some rows of the group.

A **share** is the average of a 0/1 value. In SQLite a comparison is already 0 or 1, so `AVG(license_status = 'registration number')` is the share of listings with a registration number. (In PostgreSQL write `AVG((license_status = 'registration number')::int)`.)

By hand, with the small tables of block 1: grouping the three listings by `district` gives Mitte with `COUNT(*) = 2` and `COUNT(price) = 1`, and Neukölln with 1 and 1. `AVG(price)` for Mitte is 120, not 60: the missing price is skipped. `HAVING COUNT(*) >= 2` keeps only Mitte.

### Why it matters

Almost every business question is an aggregate: listings per district, average price per room type, reviews per month, the share of registered listings. `HAVING` with a minimum count prevents tiny groups from topping a ranking by chance: a district with five listings can have any average.

### How it works in Python

Listings and average price per district. Prices are compared for **short stays** only (`minimum_nights < 28`): a third of the listings require a month or more, and their price means something else (Session 4).

In [3]:
q("""
SELECT district,
       COUNT(*) AS n_listings,
       COUNT(price) FILTER (WHERE minimum_nights < 28) AS n_short_with_price,
       ROUND(AVG(price) FILTER (WHERE minimum_nights < 28), 1) AS avg_price
FROM listings
GROUP BY district
HAVING COUNT(*) >= 300            -- filter groups, not rows
ORDER BY avg_price DESC
""")

,district,n_listings,n_short_with_price,avg_price
0,Mitte,2826,1537,220.4
1,Steglitz - Zehlendorf,356,174,217.4
2,Pankow,1950,1033,202.4
3,Treptow - Köpenick,575,333,191.0
4,Charlottenburg-Wilm.,1432,760,190.1
5,Friedrichshain-Kreuzberg,2652,1337,188.6
6,Tempelhof - Schöneberg,854,478,171.5
7,Lichtenberg,395,226,159.7
8,Neukölln,1308,556,154.9


In [4]:
q("""
SELECT room_type,
       COUNT(*) AS n,
       ROUND(AVG(license_status = 'registration number'), 3) AS share_registered,
       COUNT(*) FILTER (WHERE minimum_nights >= 28) AS n_medium_term
FROM listings
GROUP BY room_type
ORDER BY n DESC
""")

,room_type,n,share_registered,n_medium_term
0,Entire home/apt,8846,0.336,3157
1,Private room,3754,0.401,1310
2,Hotel room,89,0.000,8
3,Shared room,87,0.034,5


Hotel rooms and shared rooms (hostel beds) almost never show a registration number: the Berlin registration rule concerns private flats, while hotels and hostels are commercial businesses regulated differently, so a missing number is not a violation there. One aggregate table, read with knowledge of the domain, prevents a wrong headline.

SQLite has no function for the median. Where you need one, compute the groups' rows in SQL and the median in pandas, or build it from window functions (see the exercises).

### In practice

- **Eurostat and national statistical offices** publish tourism statistics, such as nights spent per region and month, computed from the reports of individual accommodation businesses; each published cell is a `GROUP BY` result.
- **Hospital comparisons** such as Care Compare of the US Centers for Medicare & Medicaid Services do not publish a measure when a hospital has too few cases, which is the idea of `HAVING COUNT(*) >= k`.

> **Warning.** `AVG` skips NULLs, so `AVG(price)` is the average over the listings that show a price, not over all listings (block 1, NULL values).

## Joins: INNER JOIN and LEFT JOIN

### Concept

A **join** combines rows of two tables whose key columns match. An **inner join** (`JOIN ... ON a.key = b.key`) returns one row per matching pair; rows without a partner disappear. Joining `listings` with `reviews` returns one row per listing and review month, with the district and room type of the listing added. Short **aliases** (`FROM listings AS l`) say which table a column comes from: `l.district`, `r.n_reviews`.

A **left join** (`LEFT JOIN`) keeps every row of the left table; where no partner exists, the columns of the right table are NULL. Combined with `WHERE right.key IS NULL`, it finds rows **without** a partner, an **anti-join**: for example, listings that have never been reviewed.

![Inner and left join of a small listings and reviews table](../figures/join-types.png)

### Why it matters

Normalised data must be joined before they can be analysed. The join type decides which rows survive and therefore which population a number describes. Many wrong numbers in reports come from an unintended inner join, which drops unmatched rows silently, or from a join on a key that is not unique, which repeats rows.

### How it works in Python

Reviews in 2025 per district: each review month gets the district of its listing.

In [5]:
q("""
SELECT l.district, SUM(r.n_reviews) AS reviews_2025, COUNT(DISTINCT l.id) AS listings_reviewed
FROM listings AS l
JOIN reviews AS r ON r.listing_id = l.id
WHERE r.month BETWEEN '2025-01-01' AND '2025-12-01'
GROUP BY l.district
ORDER BY reviews_2025 DESC
LIMIT 3
""")

,district,reviews_2025,listings_reviewed
0,Mitte,32541,1387
1,Friedrichshain-Kreuzberg,31179,1403
2,Pankow,21587,1035


The share of nights still available in July and August 2026 per district joins listings with the calendar. A night that is not available is either booked or blocked by the host; the calendar does not say which, so a low share is an upper bound on demand, not a measure of it.

In [6]:
q("""
SELECT l.district, COUNT(*) AS nights, ROUND(AVG(c.available), 3) AS share_available
FROM listings AS l
JOIN calendar AS c ON c.listing_id = l.id
WHERE c.date BETWEEN '2026-07-01' AND '2026-08-31'
GROUP BY l.district
ORDER BY share_available
LIMIT 3
""")

,district,nights,share_available
0,Neukölln,79532,0.257
1,Pankow,119050,0.320
2,Friedrichshain-Kreuzberg,161730,0.321


Anti-joins in both directions: listings without any review, and review rows whose listing is not in `listings` (the 57 vanished listings of block 1).

In [7]:
print(q("""
SELECT COUNT(*) AS listings_without_review
FROM listings AS l
LEFT JOIN reviews AS r ON r.listing_id = l.id
WHERE r.listing_id IS NULL
"""))
print(q("""
SELECT COUNT(*) AS review_rows, COUNT(DISTINCT r.listing_id) AS listings, SUM(r.n_reviews) AS reviews
FROM reviews AS r
LEFT JOIN listings AS l ON l.id = r.listing_id
WHERE l.id IS NULL
"""))
q("SELECT COUNT(*) AS rows_after_inner_join FROM reviews AS r JOIN listings AS l ON l.id = r.listing_id")

   listings_without_review
0                     2573
   review_rows  listings  reviews
0          499        57     4932


,rows_after_inner_join
0,226058


The inner join silently drops the 499 review rows (4,932 reviews) of the 57 vanished listings: only the comparison with the 226,557 rows of the table shows it.

### In practice

- **Marketing analysts** join transactions with customer attributes; a customer without transactions disappears in an inner join and must be kept with a left join if "inactive customers" are part of the question.
- **Epidemiological record linkage**, for example linking cancer registries with mortality registers, keeps unmatched records and analyses them separately, because who fails to link is itself informative.

> **Caution.** A join with a one-to-many relationship repeats the rows of the "one" side. After `listings JOIN reviews`, a listing with 100 review months appears 100 times, so `AVG(l.price)` is no longer the average price of a listing but an average weighted by review months: €182.2 instead of €190.2 for the reviewed short stays (computed below). Aggregate the "many" side first (next section), or compare `COUNT(*)` with `COUNT(DISTINCT key)`.

In [8]:
q("""
SELECT 'after the join' AS version, COUNT(*) AS n_rows, ROUND(AVG(l.price), 1) AS avg_price
FROM listings AS l JOIN reviews AS r ON r.listing_id = l.id
WHERE l.minimum_nights < 28
UNION ALL
SELECT 'one row per listing', COUNT(*), ROUND(AVG(price), 1)
FROM listings WHERE minimum_nights < 28 AND number_of_reviews > 0
""")

,version,n_rows,avg_price
0,after the join,206318,182.2
1,one row per listing,7728,190.2


(`UNION ALL` stacks the rows of two queries with the same columns.)

> **Warning.** A condition on the right table in `WHERE` turns a left join back into an inner join: `listings l LEFT JOIN reviews r ... WHERE r.month >= '2026-01-01'` keeps only the listings with a review in 2026, because `r.month` is NULL for all others. Put such conditions into the `ON` clause if unmatched rows must stay: `LEFT JOIN reviews r ON r.listing_id = l.id AND r.month >= '2026-01-01'` keeps all listings.

## Subqueries and common table expressions

### Concept

A **subquery** is a query in parentheses inside another query. It can return one value (`WHERE price > (SELECT AVG(price) FROM listings)`), a list (`WHERE id IN (SELECT listing_id FROM reviews WHERE ...)`) or a whole table (`FROM (SELECT ...) AS t`). `EXISTS (subquery)` is true when the subquery returns at least one row.

A **common table expression** (CTE), written `WITH name AS (SELECT ...)`, gives an intermediate result a name; later parts of the query use it like a table. Several CTEs separated by commas turn a long query into a sequence of readable steps, much like intermediate variables in Python. `WITH RECURSIVE` lets a CTE refer to itself, which repeats a step until it returns no more rows.

### Why it matters

Real questions take several steps: first aggregate the review months per listing, then compare listings; first compute a value per district, then rank the districts. Nested subqueries become hard to read after two levels; named steps can be read, tested and changed one at a time.

### How it works in Python

How many listings are more expensive than the average listing, and how many were reviewed in 2026?

In [9]:
q("""
SELECT (SELECT COUNT(*) FROM listings WHERE price > (SELECT AVG(price) FROM listings)) AS above_average,
       (SELECT COUNT(*) FROM listings
        WHERE id IN (SELECT listing_id FROM reviews WHERE month >= '2026-01-01'))   AS reviewed_in_2026
""")

,above_average,reviewed_in_2026
0,3257,6009


The same question as the caution above, done correctly with a CTE: first one row per listing (reviews summed), then the join, so that every listing counts once.

In [10]:
q("""
WITH per_listing AS (                       -- step 1: one row per listing
    SELECT listing_id, SUM(n_reviews) AS n_reviews, MIN(month) AS first_month
    FROM reviews
    GROUP BY listing_id
)
SELECT l.room_type,                          -- step 2: join and aggregate
       COUNT(*) AS listings,
       ROUND(AVG(l.price), 1) AS avg_price,
       ROUND(AVG(p.n_reviews), 1) AS avg_reviews
FROM listings AS l
JOIN per_listing AS p ON p.listing_id = l.id
WHERE l.minimum_nights < 28
GROUP BY l.room_type
ORDER BY listings DESC
""")

,room_type,listings,avg_price,avg_reviews
0,Entire home/apt,5348,222.0,84.3
1,Private room,2225,113.1,74.7
2,Shared room,78,56.9,134.6
3,Hotel room,77,200.8,130.2


### In practice

- **Analysts** write long reports as chains of CTEs, one per step, and check each step on its own by selecting from it.
- **Hierarchies**, such as an employee's chain of managers or the customs nomenclature from chapter to heading in the final project, are walked with recursive CTEs.

> **Caution.** `WHERE id NOT IN (SELECT listing_id FROM ...)` behaves unexpectedly when the subquery returns a NULL (remember the three-valued logic of block 1). Find rows without a partner with `NOT EXISTS (...)` or with a left join and `IS NULL`.

## Window functions: ranking, running totals, comparison with the previous row

### Concept

A **window function** computes a value for each row from a set of related rows, the **window**, without collapsing them into one row. This is the difference from `GROUP BY`, which returns one row per group. The window is defined after `OVER`:

- `PARTITION BY room_type` restricts the window to rows of the same room type (like a group);
- `ORDER BY avg_price DESC` sorts the rows inside the window;
- the function decides what is computed: `ROW_NUMBER()` numbers the rows 1, 2, 3, ...; `RANK()` gives ties the same number and leaves a gap; `DENSE_RANK()` gives ties the same number without a gap; `NTILE(k)` splits the rows into *k* groups of nearly equal size; `SUM(n) OVER (ORDER BY month)` is a **running total**; `AVG(x) OVER (PARTITION BY district)` puts the district average next to each row; `LAG(n)` returns the value of the previous row, `LAG(n, 12)` the value twelve rows back.

Window functions are evaluated after `WHERE`, `GROUP BY` and `HAVING`. A filter on their result, such as "rank 1 only", must therefore be placed one query level higher, in the next CTE or in the outer query.

By hand. Average prices per room type and district (invented numbers):

| room_type | district | avg_price | `RANK() OVER (PARTITION BY room_type ORDER BY avg_price DESC)` | `DENSE_RANK()` | `ROW_NUMBER()` |
|---|---|---|---|---|---|
| Entire home/apt | Mitte | 210 | 1 | 1 | 1 |
| Entire home/apt | Pankow | 210 | 1 | 1 | 2 |
| Entire home/apt | Neukölln | 160 | 3 | 2 | 3 |
| Private room | Mitte | 115 | 1 | 1 | 1 |
| Private room | Neukölln | 95 | 2 | 2 | 2 |

The numbering restarts for each room type (partition). Mitte and Pankow tie for entire homes; `RANK` then skips to 3, `DENSE_RANK` continues with 2, and `ROW_NUMBER` breaks the tie arbitrarily.

```mermaid
flowchart TB
    T["listings<br/>12,776 rows"] --> S["CTE short_stays:<br/>WHERE minimum_nights &lt; 28"]
    S --> G["CTE per_district:<br/>GROUP BY room_type, district<br/>HAVING COUNT(*) &gt;= 30"]
    G --> W["CTE ranked:<br/>RANK() OVER (PARTITION BY<br/>room_type ORDER BY avg_price DESC)"]
    W --> F["outer query:<br/>WHERE rnk &lt;= 3"]
```

### Why it matters

Rankings within groups, running totals, changes against the previous period, moving averages and "this row compared with its group" are standard analytical questions. Without window functions they need self-joins or loops in Python; with them they are one readable query that the database executes over all rows.

### How it works in Python

The three most expensive districts for each room type, by average short-stay price (the query is also in [`sql/02-rank-districts-by-price.sql`](sql/02-rank-districts-by-price.sql)):

In [11]:
q("""
WITH short_stays AS (                      -- step 1: the listings the question is about
    SELECT district, room_type, price
    FROM listings
    WHERE minimum_nights < 28 AND price IS NOT NULL
      AND room_type IN ('Entire home/apt', 'Private room')
), per_district AS (                       -- step 2: one row per room type and district
    SELECT room_type, district, COUNT(*) AS n_listings, ROUND(AVG(price), 1) AS avg_price
    FROM short_stays
    GROUP BY room_type, district
    HAVING COUNT(*) >= 30
), ranked AS (                             -- step 3: rank inside each room type
    SELECT *, RANK() OVER (PARTITION BY room_type ORDER BY avg_price DESC) AS rnk
    FROM per_district
)
SELECT room_type, rnk, district, n_listings, avg_price
FROM ranked
WHERE rnk <= 3                             -- filter one level up
ORDER BY room_type, rnk
""")

,room_type,rnk,district,n_listings,avg_price
0,Entire home/apt,1,Spandau,61,356.1
1,Entire home/apt,2,Steglitz - Zehlendorf,115,281.7
2,Entire home/apt,3,Mitte,1116,247.1
3,Private room,1,Mitte,367,154.9
4,Private room,2,Charlottenburg-Wilm.,278,125.3
5,Private room,3,Pankow,199,111.7


Spandau leads for entire homes with only 61 listings. One of them asks €7,999.20 a night, which is enough to lift the average of a small group; ranked by the median price, Mitte leads both room types. The `HAVING` threshold protects against tiny groups, not against single extreme values; Session 4 treats such outliers.

Running totals and month-to-month changes of the reviews, the course's proxy for demand, around the first COVID-19 lockdown:

In [12]:
q("""
WITH monthly AS (
    SELECT month, SUM(n_reviews) AS n
    FROM reviews
    WHERE month BETWEEN '2019-12-01' AND '2020-06-01'
    GROUP BY month
)
SELECT month, n,
       SUM(n) OVER (ORDER BY month)                         AS running_total,
       n - LAG(n) OVER (ORDER BY month)                     AS change,
       ROUND(AVG(n) OVER (ORDER BY month ROWS 2 PRECEDING)) AS moving_avg_3m
FROM monthly
ORDER BY month
""")

,month,n,running_total,change,moving_avg_3m
0,2019-12-01,4070,4070,NaN,4070.0
1,2020-01-01,3971,8041,-99.0,4021.0
2,2020-02-01,4055,12096,84.0,4032.0
3,2020-03-01,2512,14608,-1543.0,3513.0
4,2020-04-01,285,14893,-2227.0,2284.0
5,2020-05-01,612,15505,327.0,1136.0
6,2020-06-01,1547,17052,935.0,815.0


`LAG` has no previous row for December, so the change is NULL (shown as `NaN` in pandas). The moving average uses fewer than three months at the start (`ROWS 2 PRECEDING`: the current row and the two before it). In April 2020, Berlin's listings received 285 reviews instead of about 4,000 (Session 12 models this break).

![Reviews per year 2012-2025 with the year-over-year change computed by a window function](../figures/reviews-per-year-window.png)

A window function can also place each listing next to its group without collapsing the rows. Here the share of available nights in the coming year per listing, compared with the district average:

In [13]:
q("""
WITH per_listing AS (                         -- one row per listing: share of available nights
    SELECT l.id, l.district, AVG(c.available) AS share_free
    FROM listings AS l JOIN calendar AS c ON c.listing_id = l.id
    GROUP BY l.id, l.district
), compared AS (                              -- window: each listing next to its district
    SELECT *, AVG(share_free) OVER (PARTITION BY district) AS district_avg
    FROM per_listing
)
SELECT district,
       ROUND(MIN(district_avg), 3) AS avg_share_free,
       ROUND(AVG(share_free = 0), 3) AS share_never_available,
       COUNT(*) FILTER (WHERE share_free - district_avg > 0.5) AS far_freer_than_district
FROM compared
GROUP BY district
ORDER BY avg_share_free
LIMIT 3
""")

,district,avg_share_free,share_never_available,far_freer_than_district
0,Neukölln,0.283,0.427,203
1,Friedrichshain-Kreuzberg,0.378,0.319,441
2,Tempelhof - Schöneberg,0.384,0.246,113


In Neukölln, 43 % of the listings have not a single available night in the coming year. That is not a sign of booming demand: a calendar that is closed for a whole year usually means the listing is not offered at all.

### In practice

- **Finance teams** compute month-over-month growth and cumulative revenue with `LAG` and running sums for management reports.
- **Health-services research** with claims data uses `LAG` over the admissions of each patient to measure the time to readmission, the basis of readmission indicators.
- In pandas, `groupby(...).rank()` corresponds to `RANK() OVER (PARTITION BY ...)`, `cumsum()` to a running `SUM`, `diff()` to `n - LAG(n)`, and `groupby(...).transform("mean")` to `AVG(...) OVER (PARTITION BY ...)`.

> **Warning.** `RANK`, `DENSE_RANK` and `ROW_NUMBER` differ only when there are ties, so a bug may go unnoticed for a long time. Decide explicitly how ties should be handled, and add a second sort key (for example `ORDER BY avg_price DESC, district`) when you need a reproducible order.

> **Caution.** `WHERE RANK() OVER (...) = 1` is an error: `WHERE` is evaluated before window functions. Compute the rank in a CTE and filter in the next step.

## Running SQL from Python

### Concept

The module `sqlite3` is part of Python's standard library. Four objects and ideas cover almost all work:

- **Connection.** `con = sqlite3.connect("file.sqlite")` opens the file (and creates it if it does not exist). `con.close()` closes it.
- **Cursor.** `cur = con.cursor()` creates a cursor, the object that executes a statement and walks through its result: `cur.execute(sql)`, then `cur.fetchone()` (the next row as a tuple), `cur.fetchall()` (all rows as a list of tuples) or a `for` loop over `cur`. `con.execute(sql)` is a shortcut that creates the cursor for you.
- **Parameters.** Values that change between runs are passed separately from the SQL text, with placeholders: `?` filled from a tuple, or `:name` filled from a dictionary. SQLite receives the values as data, so they can never be read as SQL.
- **Transactions.** Statements that change data (`INSERT`, `UPDATE`, `DELETE`) are kept in a transaction until `con.commit()`; `con.rollback()` undoes them. `with con: ...` commits at the end of the block, or rolls back if an error occurs (it does not close the connection).

pandas adds two functions: `pd.read_sql_query(sql, con, params=...)` runs a query and returns a DataFrame; `DataFrame.to_sql(name, con)` writes a DataFrame into a table.

```mermaid
sequenceDiagram
    participant P as Python
    participant C as sqlite3 connection
    participant F as SQLite file
    P->>C: execute(sql, parameters)
    C->>F: read and write pages of the file
    F-->>C: result rows
    C-->>P: tuples (fetchall) or a DataFrame (read_sql_query)
```

### Why it matters

The database does the filtering, joining and aggregation on all rows; Python receives a small result for plotting and modelling, and a script can repeat the whole analysis with one command. Parameters protect against **SQL injection**, an attack in which text entered by a user changes the meaning of a query: with string formatting, a "district" such as `x' OR '1'='1` turns the condition into one that is true for every row.

### How it works in Python

In [14]:
cur = con.cursor()
cur.execute("SELECT district, COUNT(*) FROM listings GROUP BY district ORDER BY 2 DESC")
print(cur.fetchone())          # the first row, as a tuple
print(cur.fetchall()[:2])      # the remaining rows, as a list of tuples (first two shown)

# parameters: ? from a tuple, :name from a dictionary
print(con.execute("SELECT COUNT(*) FROM listings WHERE district = ? AND license_status = ?",
                  ("Neukölln", "registration number")).fetchone())
print(con.execute("SELECT COUNT(*) FROM listings WHERE district = :d AND price < :p",
                  {"d": "Neukölln", "p": 60}).fetchone())

('Mitte', 2826)
[('Friedrichshain-Kreuzberg', 2652), ('Pankow', 1950)]
(541,)
(148,)


What an f-string does with input from a user, and what a parameter does with the same input:

In [15]:
user_input = "x' OR '1'='1"
unsafe = f"SELECT COUNT(*) FROM listings WHERE district = '{user_input}'"
print(unsafe)
print("f-string: ", con.execute(unsafe).fetchone())                                                  # every row
print("parameter:", con.execute("SELECT COUNT(*) FROM listings WHERE district = ?", (user_input,)).fetchone())

SELECT COUNT(*) FROM listings WHERE district = 'x' OR '1'='1'
f-string:  (12776,)
parameter: (0,)


The f-string turned the input into SQL, and the condition became true for all 12,776 listings; the parameter was treated as a district name, and no district has that name.

A query result as a DataFrame, with parameters:

In [16]:
by_room = q("""
SELECT room_type, COUNT(*) AS n_listings, ROUND(AVG(price), 1) AS avg_price
FROM listings
WHERE district = ? AND minimum_nights < ? AND price IS NOT NULL
GROUP BY room_type ORDER BY n_listings DESC
""", params=("Neukölln", 28))
by_room

,room_type,n_listings,avg_price
0,Entire home/apt,350,185.7
1,Private room,199,104.5
2,Shared room,7,50.9


Writing data. A transaction with `with con:` creates a small table of district targets in a temporary database, inserts rows with `executemany` and commits; the error in the second block rolls everything in that block back.

In [17]:
scratch = sqlite3.connect(":memory:")
with scratch:                                               # commit at the end of the block
    scratch.execute("CREATE TABLE targets (district TEXT PRIMARY KEY, max_listings INTEGER NOT NULL)")
    scratch.executemany("INSERT INTO targets VALUES (?, ?)", [("Mitte", 2500), ("Pankow", 1800)])
try:
    with scratch:                                           # roll back if anything fails
        scratch.execute("INSERT INTO targets VALUES (?, ?)", ("Neukölln", 1200))
        scratch.execute("INSERT INTO targets VALUES (?, ?)", ("Mitte", 1000))     # duplicate key
except sqlite3.IntegrityError as error:
    print("rolled back:", error)
print(scratch.execute("SELECT * FROM targets").fetchall())   # Neukölln is not there either
scratch.close()

rolled back: UNIQUE constraint failed: targets.district
[('Mitte', 2500), ('Pankow', 1800)]


**A script that answers a question.** Notebooks are good for exploring; a question that is asked again and again belongs in a script (Session 2). The course has two small scripts:

- [`make_airbnb_sqlite.py`](make_airbnb_sqlite.py) **loads data**: it reads the Parquet files, creates `listings` with its primary key, writes the three tables with `to_sql`, commits and prints the row counts as a check.
- [`district_prices.py`](district_prices.py) **answers a question** for any district given on the command line. Its core:

```python
QUERY = """
SELECT room_type, COUNT(*) AS n_listings, ROUND(AVG(price), 1) AS avg_price
FROM listings
WHERE district = ? AND minimum_nights < ? AND price IS NOT NULL
GROUP BY room_type
ORDER BY n_listings DESC
"""


def district_prices(con: sqlite3.Connection, district: str, max_nights: int = 28) -> pd.DataFrame:
    """Short-stay listings and average price per room type in one district."""
    return pd.read_sql_query(QUERY, con, params=(district, max_nights))
```

The `main()` function reads the district from the command line with `argparse`, checks it against the districts in the table, opens the connection, calls the function and closes the connection in a `finally` block. In a terminal, from the repository root: `uv run python sessions/03-sql/workbooks/district_prices.py Pankow`. The cell below runs it from the notebook.

In [18]:
result = subprocess.run([sys.executable, "sessions/03-sql/workbooks/district_prices.py", "Pankow"],
                        capture_output=True, text=True, check=True)
print(result.stdout)

Short stays (minimum stay below 28 nights) with a price in Pankow:
      room_type  n_listings  avg_price
Entire home/apt         816      224.7
   Private room         199      111.7
     Hotel room          15      218.0
    Shared room           3       68.0



### In practice

- **Analysts** in most companies pull aggregated data from a data warehouse into notebooks with exactly this pattern; the heavy work stays in the database. With a server database, only the connection line changes (for PostgreSQL, for example, with the packages `psycopg` or SQLAlchemy).
- **SQL injection** belongs to the "Injection" category of the OWASP Top 10 web application security risks.
- **Scheduled scripts** (a nightly job, a GitHub Actions workflow) load new data and recompute the same report every day, which is impossible with manual steps.

> **Caution.** Never build SQL with f-strings from values that come from users, files or the web. Use parameters. Table and column names cannot be parameters; if they must vary, check them against a fixed list, as `district_prices.py` checks the district.

> **Tip.** `pd.read_sql_query` loads the whole result into memory. Aggregate in SQL first: `SELECT * FROM calendar` sends 4.7 million rows to Python, while the share of available nights per district is twelve rows.

## Check your understanding

1. Explain the difference between `WHERE` and `HAVING` with an example from the listings.
2. A colleague joins `listings` with `reviews` using an inner join and reports the number of listings per district. Which listings are missing from the result, and does it matter for her question?
3. Rewrite "the three cheapest districts for private rooms" as CTEs and an outer query. In which step does the window function appear, and in which step the filter?
4. When do `RANK()` and `ROW_NUMBER()` give different results?
5. Why is `con.execute("... WHERE district = ?", (d,))` safer than an f-string, and what happens to new rows if a script ends without `commit()`?

## Further reading

- SQLite. *Window functions*. https://www.sqlite.org/windowfunctions.html
- SQLite. *The WITH clause* (common table expressions). https://www.sqlite.org/lang_with.html
- Python documentation. *sqlite3: DB-API 2.0 interface for SQLite databases* (tutorial and how-to guides). https://docs.python.org/3/library/sqlite3.html
- pandas documentation. *Comparison with SQL*. https://pandas.pydata.org/docs/getting_started/comparison/comparison_with_sql.html
- The PostgreSQL Global Development Group. *Tutorial: window functions*. https://www.postgresql.org/docs/current/tutorial-window.html (the same SQL on a server database)
- OWASP. *SQL injection prevention cheat sheet*. https://cheatsheetseries.owasp.org/cheatsheets/SQL_Injection_Prevention_Cheat_Sheet.html